# Calculation Demonstration: Step-by-Step Overshoot Walkthrough

This notebook walks through **every input variable and formula** used by `scripts/calculate_overshoot.py`,
using **USA (area_code = 231)** and **year 2023** as a worked example.

---

### Overview

The Ecological Footprint (EF) measures how much biologically productive area a country requires to
produce the resources it consumes and absorb the CO\u2082 waste it generates. Biocapacity (BC) measures
how much productive area the country actually has.

**6 EF Components:** Cropland, Grazing, Forest, Fishing, Built-up, Carbon

**5 BC Components:** Cropland, Grazing, Forest, Fishing, Built-up (no BC for carbon\u2014forest BC already covers sequestration capacity)

**Demo Country:** United States of America (area_code = 231) 
**Demo Year:** 2023

Each section below:
1. Shows the formula in markdown
2. Loads and displays the raw input data
3. Performs the step-by-step calculation
4. Compares the result to `overshoot_results.csv` for validation

---
## Section 1: Load Data & Reference Parameters

In [1]:
import os
import zipfile
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.max_columns", 25)
pd.set_option("display.max_rows", 60)

# ── Paths ───────────────────────────────────────────────────────────────────
ROOT = Path(".").resolve().parent
DATA = ROOT / "data"

DEMO_COUNTRY = 231       # USA
DEMO_YEAR = 2023
YEAR_MIN, YEAR_MAX = 2014, 2023

def load(rel_path, year_filter=True):
    """Load a FAOSTAT CSV, optionally filter to 2014-2023."""
    df = pd.read_csv(DATA / rel_path)
    if year_filter and "Year" in df.columns:
        df = df[df["Year"].between(YEAR_MIN, YEAR_MAX)]
    return df

# ── 8 input data files ───────────────────────────────────────────────────────
crop_prod    = load("01_cropland/crop_production.csv")
cropland_area = load("01_cropland/cropland_area.csv")
livestock    = load("02_grazing/livestock_stocks.csv")
pasture_area = load("02_grazing/pasture_area.csv")
timber       = load("03_forest/timber_production.csv")
forest_area  = load("03_forest/forest_area.csv")
fish_fbs     = load("04_fishing/fish_supply_fbs.csv")
co2_energy   = load("06_carbon/co2_energy_faostat.csv")
population   = load("07_population/population.csv")

# Land-use ZIP
land_use_zip = DATA / "raw" / "Inputs_LandUse_E_All_Data_(Normalized).zip"
with zipfile.ZipFile(land_use_zip) as z:
    csv_name = [n for n in z.namelist() if n.endswith(".csv") and "Data" in n][0]
    land_use_all = pd.read_csv(z.open(csv_name), encoding="latin-1")
land_use_all = land_use_all[land_use_all["Year"].between(YEAR_MIN, YEAR_MAX)]

# Results for validation
results = pd.read_csv(DATA / "overshoot_results.csv")
global_summary = pd.read_csv(DATA / "overshoot_global_summary.csv")

# Helper: get stored result for a country+year
def stored(area_code, year, col):
    row = results[(results["area_code"] == area_code) & (results["year"] == year)]
    return row[col].iloc[0] if len(row) > 0 else None

print("All data loaded.")

All data loaded.


### Reference Parameter CSVs

These CSV files contain the static parameters used throughout the calculation.

In [2]:
print("=" * 70)
print("EQUIVALENCE FACTORS (gha per ha)")
print("=" * 70)
eqf_df = pd.read_csv(DATA / "09_factors" / "equivalence_factors_reference_1999.csv")
display(eqf_df)

print("\n" + "=" * 70)
print("GLEAM FEED COEFFICIENTS (pasture DM demand per head)")
print("=" * 70)
gleam_df = pd.read_csv(DATA / "02_grazing" / "gleam_feed_coefficients.csv")
display(gleam_df)

print("\n" + "=" * 70)
print("PPR CALCULATION PARAMETERS")
print("=" * 70)
ppr_df = pd.read_csv(DATA / "04_fishing" / "ppr_calculation_parameters.csv")
display(ppr_df)

print("\n" + "=" * 70)
print("CARBON SEQUESTRATION PARAMETERS")
print("=" * 70)
carbon_params_df = pd.read_csv(DATA / "10_reference_parameters" / "carbon_sequestration_parameters.csv")
display(carbon_params_df)

print("\n" + "=" * 70)
print("FOREST NAI REFERENCE")
print("=" * 70)
nai_df = pd.read_csv(DATA / "10_reference_parameters" / "forest_nai_reference.csv")
display(nai_df)

EQUIVALENCE FACTORS (gha per ha)


,land_type,equivalence_factor_gha_per_ha,notes
0,Cropland,2.1000,Wackernagel et al. 2002 (Table 1)
1,Built-up land,2.2000,Assumed same productivity as cropland it replaces
2,Forest,1.3000,Wackernagel et al. 2002 (Table 1)
3,Grazing land,0.5000,Wackernagel et al. 2002 (Table 1)
4,Fishing grounds,0.4000,Wackernagel et al. 2002 (Table 1)
5,Carbon (fossil fuel),1.3000,Uses forest equivalence factor (sequestration ...



GLEAM FEED COEFFICIENTS (pasture DM demand per head)


,animal_type,total_dmi_kg_per_day,pasture_fraction,crop_fraction,residue_fraction,pasture_dmi_kg_per_day,annual_pasture_demand_t_dm,source
0,Cattle (dairy),14.5000,0.4500,0.3000,0.2500,6.5300,2.3820,FAO GLEAM 3.0 (global averages)
1,Cattle (beef),9.8000,0.6500,0.1500,0.2000,6.3700,2.3250,FAO GLEAM 3.0 (global averages)
2,Buffalo,12.0000,0.5500,0.2000,0.2500,6.6000,2.4090,FAO GLEAM 3.0 (global averages)
3,Sheep,1.8000,0.7000,0.1000,0.2000,1.2600,0.4600,FAO GLEAM 3.0 (global averages)
4,Goats,1.5000,0.7200,0.0800,0.2000,1.0800,0.3940,FAO GLEAM 3.0 (global averages)
5,Pigs,2.5000,0.0000,0.7500,0.2500,0.0000,0.0000,FAO GLEAM 3.0 (global averages)
6,Chickens (layers),0.1200,0.0000,0.9000,0.1000,0.0000,0.0000,FAO GLEAM 3.0 (global averages)
7,Chickens (broilers),0.1000,0.0000,0.9000,0.1000,0.0000,0.0000,FAO GLEAM 3.0 (global averages)



PPR CALCULATION PARAMETERS


,parameter,value,unit,source,notes
0,transfer_efficiency,0.1013,fraction,Pauly & Christensen 1995,Efficiency between trophic levels (10.13%)
1,discard_rate,1.2700,ratio,Pauly & Christensen 1995,1.27 = 0.27 t bycatch per tonne harvested
2,wet_weight_to_carbon,9.0000,ratio,Sea Around Us / Pauly,9:1 wet weight to carbon conversion
3,sustainable_catch_global,93.0000,million tonnes/yr,Gulland 1971 / FAO,Maximum sustainable yield for global marine ca...
4,continental_shelf_area,"2,000,000,000.0000",hectares,Various oceanographic,~2 billion ha of continental shelf



CARBON SEQUESTRATION PARAMETERS


,parameter,value,uncertainty,unit,source,notes
0,AFCS,0.7300,± 0.37,tonnes C / ha / year,Mancini et al. (NFA 2016),Average Forest-Carbon Sequestration rate for w...
1,ocean_co2_fraction_min,0.2800,NaN,fraction,Khatiwala et al. 2009 (Nature),Minimum ocean absorption of anthropogenic CO2 ...
2,ocean_co2_fraction_max,0.3500,NaN,fraction,Khatiwala et al. 2009 (Nature),Maximum ocean absorption of anthropogenic CO2 ...
3,ocean_co2_fraction_default,0.3500,NaN,fraction,Wackernagel et al. 2002 / IPCC 2001,Commonly used constant for ocean CO2 absorptio...



FOREST NAI REFERENCE


,parameter,value,unit,source,notes
0,world_avg_forest_NAI,1.8100,m3 harvestable wood / ha / year,UNECE/FAO TBFRA 2000 + FAO GFSM 1998,Net Annual Increment. Denominator for forest p...


In [3]:
# ── All constants used in calculate_overshoot.py ────────────────────────────
EQF_CROPLAND = 2.1
EQF_BUILT_UP = 2.2
EQF_FOREST   = 1.3
EQF_GRAZING  = 0.5
EQF_FISHING  = 0.4
EQF_CARBON   = 1.3

GRASSLAND_NPP = 2.45
FOREST_NAI    = 1.81
AFCS          = 0.73
OCEAN_CO2_FRAC = 0.35

TRANSFER_EFFICIENCY = 0.1013
DISCARD_RATE        = 1.27
WET_TO_CARBON       = 9.0
SUSTAINABLE_CATCH_MT = 93.0
CONTINENTAL_SHELF_HA = 2_000_000_000

TROPHIC_LEVELS = {
    "Freshwater Fish": 2.5, "Demersal Fish": 3.8, "Pelagic Fish": 3.0,
    "Marine Fish, Other": 3.2, "Crustaceans": 2.5, "Cephalopods": 3.5,
    "Molluscs, Other": 2.1, "Aquatic Animals, Others": 3.0,
    "Aquatic Products, Other": 3.0, "Meat, Aquatic Mammals": 3.2,
}

GLEAM_PASTURE = {
    "Cattle": 2.35, "Buffalo": 2.409, "Sheep": 0.46,
    "Goats": 0.394, "Camels": 2.0, "Swine / pigs": 0.0,
}

constants_summary = pd.DataFrame([
    ["EQF_CROPLAND", 2.1, "gha/ha", "Wackernagel et al. 2002"],
    ["EQF_BUILT_UP", 2.2, "gha/ha", "Assumes cropland productivity"],
    ["EQF_FOREST", 1.3, "gha/ha", "Wackernagel et al. 2002"],
    ["EQF_GRAZING", 0.5, "gha/ha", "Wackernagel et al. 2002"],
    ["EQF_FISHING", 0.4, "gha/ha", "Wackernagel et al. 2002"],
    ["EQF_CARBON", 1.3, "gha/ha", "Uses forest EQF"],
    ["GRASSLAND_NPP", 2.45, "t DM/ha/yr", "Monfreda et al. 2008"],
    ["FOREST_NAI", 1.81, "m\u00b3/ha/yr", "UNECE/FAO TBFRA 2000"],
    ["AFCS", 0.73, "t C/ha/yr", "Mancini et al. 2016"],
    ["OCEAN_CO2_FRAC", 0.35, "fraction", "Khatiwala et al. 2009"],
    ["TRANSFER_EFFICIENCY", 0.1013, "fraction", "Pauly & Christensen 1995"],
    ["DISCARD_RATE", 1.27, "ratio", "Pauly & Christensen 1995"],
    ["WET_TO_CARBON", 9.0, "ratio", "Sea Around Us / Pauly"],
    ["SUSTAINABLE_CATCH", 93.0, "Mt/yr", "Gulland 1971 / FAO"],
    ["CONTINENTAL_SHELF", 2e9, "ha", "Various oceanographic"],
], columns=["Constant", "Value", "Unit", "Source"])

print("\n" + "=" * 70)
print("ALL HARD-CODED CONSTANTS")
print("=" * 70)
display(constants_summary)


ALL HARD-CODED CONSTANTS


,Constant,Value,Unit,Source
0,EQF_CROPLAND,2.1000,gha/ha,Wackernagel et al. 2002
1,EQF_BUILT_UP,2.2000,gha/ha,Assumes cropland productivity
2,EQF_FOREST,1.3000,gha/ha,Wackernagel et al. 2002
3,EQF_GRAZING,0.5000,gha/ha,Wackernagel et al. 2002
4,EQF_FISHING,0.4000,gha/ha,Wackernagel et al. 2002
5,EQF_CARBON,1.3000,gha/ha,Uses forest EQF
6,GRASSLAND_NPP,2.4500,t DM/ha/yr,Monfreda et al. 2008
7,FOREST_NAI,1.8100,m³/ha/yr,UNECE/FAO TBFRA 2000
8,AFCS,0.7300,t C/ha/yr,Mancini et al. 2016
9,OCEAN_CO2_FRAC,0.3500,fraction,Khatiwala et al. 2009


---
## Section 2: Input Data Overview

For each of the 8 FAOSTAT input files: shape, columns, year range, and sample rows filtered to USA 2023.

**Key filter:** `Area Code < 5000` selects individual countries (codes \u2265 5000 are FAOSTAT aggregates like \"World\", \"Africa\", etc.).

In [4]:
datasets = [
    ("crop_production.csv", crop_prod),
    ("cropland_area.csv", cropland_area),
    ("livestock_stocks.csv", livestock),
    ("pasture_area.csv", pasture_area),
    ("timber_production.csv", timber),
    ("forest_area.csv", forest_area),
    ("fish_supply_fbs.csv", fish_fbs),
    ("co2_energy_faostat.csv", co2_energy),
]

for name, df in datasets:
    print(f"\n{'=' * 70}")
    print(f"{name}")
    print(f"{'=' * 70}")
    print(f"Shape: {df.shape}  |  Columns: {list(df.columns)}")
    print(f"Years: {df['Year'].min()}-{df['Year'].max()}  |  Countries (Area Code < 5000): {df[df['Area Code'] < 5000]['Area Code'].nunique()}")
    usa = df[(df['Area Code'] == DEMO_COUNTRY) & (df['Year'] == DEMO_YEAR)]
    print(f"USA 2023 rows: {len(usa)}")
    display(usa.head(3))


crop_production.csv
Shape: (26273, 14)  |  Columns: ['Area Code', 'Area Code (M49)', 'Area', 'Item Code', 'Item Code (CPC)', 'Item', 'Element Code', 'Element', 'Year Code', 'Year', 'Unit', 'Value', 'Flag', 'Note']


Years: 2014-2023  |  Countries (Area Code < 5000): 198
USA 2023 rows: 12


,Area Code,Area Code (M49),Area,Item Code,Item Code (CPC),Item,Element Code,Element,Year Code,Year,Unit,Value,Flag,Note
128387,231,'840,United States of America,1717,'F1717,"Cereals, primary",5312,Area harvested,2023,2023,ha,"55,537,673.0000",A,NaN
128451,231,'840,United States of America,1717,'F1717,"Cereals, primary",5510,Production,2023,2023,t,"462,631,157.5500",A,NaN
128515,231,'840,United States of America,1738,'F1738,Fruit Primary,5312,Area harvested,2023,2023,ha,"1,039,118.0000",A,NaN



cropland_area.csv
Shape: (7695, 13)  |  Columns: ['Area Code', 'Area Code (M49)', 'Area', 'Item Code', 'Item', 'Element Code', 'Element', 'Year Code', 'Year', 'Unit', 'Value', 'Flag', 'Note']
Years: 2014-2023  |  Countries (Area Code < 5000): 228
USA 2023 rows: 3


,Area Code,Area Code (M49),Area,Item Code,Item,Element Code,Element,Year Code,Year,Unit,Value,Flag,Note
36651,231,'840,United States of America,6610,Agricultural land,5110,Area,2023,2023,1000 ha,"421,540.3674",I,NaN
36714,231,'840,United States of America,6621,Arable land,5110,Area,2023,2023,1000 ha,"151,563.5251",I,NaN
36777,231,'840,United States of America,6650,Permanent crops,5110,Area,2023,2023,1000 ha,"3,170.6000",I,NaN


...


fish_supply_fbs.csv
Shape: (318114, 14)  |  Columns: ['Area Code', 'Area Code (M49)', 'Area', 'Item Code', 'Item Code (FBS)', 'Item', 'Element Code', 'Element', 'Year Code', 'Year', 'Unit', 'Value', 'Flag', 'Note']


Years: 2014-2023  |  Countries (Area Code < 5000): 179
USA 2023 rows: 184


,Area Code,Area Code (M49),Area,Item Code,Item Code (FBS),Item,Element Code,Element,Year Code,Year,Unit,Value,Flag,Note
136332,231,'840,United States of America,2781,'S2781,"Fish, Body Oil",5511,Production,2023,2023,1000 t,85.0000,E,NaN
137606,231,'840,United States of America,2781,'S2781,"Fish, Body Oil",5611,Import quantity,2023,2023,1000 t,20.0000,E,NaN
138864,231,'840,United States of America,2781,'S2781,"Fish, Body Oil",5072,Stock Variation,2023,2023,1000 t,0.0000,E,NaN



co2_energy_faostat.csv
Shape: (2430, 15)  |  Columns: ['Area Code', 'Area Code (M49)', 'Area', 'Item Code', 'Item', 'Element Code', 'Element', 'Year Code', 'Year', 'Source Code', 'Source', 'Unit', 'Value', 'Flag', 'Note']
Years: 2014-2023  |  Countries (Area Code < 5000): 205
USA 2023 rows: 1


,Area Code,Area Code (M49),Area,Item Code,Item,Element Code,Element,Year Code,Year,Source Code,Source,Unit,Value,Flag,Note
11612,231,'840,United States of America,6821,Energy,7273,Emissions (CO2),2023,2023,3050,FAO TIER 1,kt,"4,640,000.0000",E,NaN


---
## Section 3: Cropland EF \u2014 Step by Step

**Formula:**
$$\text{EF}_{\text{cropland}} = \sum_{\text{crops}} \frac{\text{Production}_t}{\text{WorldYield}_{t/ha}} \times \text{EQF}_{\text{cropland}} (2.1)$$

For each crop:
1. Get national production (tonnes) from `crop_production.csv`
2. Compute world yield = world production / world harvested area (summing all countries with Area Code < 5000)
3. Divide national production by world yield \u2192 hectare-equivalents
4. Sum across all crops, multiply by EQF

In [5]:
# Step 1: USA 2023 crop production (top 10 by production tonnes)
usa_prod = crop_prod[
    (crop_prod["Area Code"] == DEMO_COUNTRY)
    & (crop_prod["Year"] == DEMO_YEAR)
    & (crop_prod["Element"] == "Production")
].copy()
usa_prod = usa_prod.sort_values("Value", ascending=False)

print(f"USA 2023: {len(usa_prod)} crop items with production data")
print(f"\nTop 10 crops by production (tonnes):")
display(usa_prod[["Item Code", "Item", "Value", "Unit"]].head(10))

USA 2023: 6 crop items with production data

Top 10 crops by production (tonnes):


,Item Code,Item,Value,Unit
128451,1717,"Cereals, primary","462,631,157.5500",t
129091,1735,Vegetables Primary,"31,253,511.2700",t
128707,1732,"Oilcrops, Oil Equivalent","23,240,551.3100",t
128579,1738,Fruit Primary,"21,876,188.6900",t
128963,1720,"Roots and Tubers, Total","21,131,470.4800",t
128835,1726,"Pulses, Total","2,367,610.0000",t


In [6]:
# Step 2: Compute world yield per crop (world production / world harvested area)
_cp = crop_prod[crop_prod["Area Code"] < 5000].copy()
_cp_prod = (
    _cp[_cp["Element"] == "Production"]
    .groupby(["Item Code", "Year"])["Value"].sum()
    .rename("world_production_t")
)
_cp_area = (
    _cp[_cp["Element"] == "Area harvested"]
    .groupby(["Item Code", "Year"])["Value"].sum()
    .rename("world_area_ha")
)
world_crop_yield = pd.concat([_cp_prod, _cp_area], axis=1)
world_crop_yield["world_yield_t_per_ha"] = (
    world_crop_yield["world_production_t"] / world_crop_yield["world_area_ha"]
)
world_crop_yield["world_yield_t_per_ha"] = (
    world_crop_yield["world_yield_t_per_ha"].replace([np.inf, -np.inf], 0).fillna(0)
)

# Show world yield for USA's top 10 crops
top10_items = usa_prod["Item Code"].head(10).tolist()
wy_2023 = world_crop_yield.loc[
    world_crop_yield.index.get_level_values("Year") == DEMO_YEAR
].copy()
wy_top10 = wy_2023.loc[wy_2023.index.get_level_values("Item Code").isin(top10_items)]

print("World yield for USA's top 10 crops (2023):")
display(wy_top10)

World yield for USA's top 10 crops (2023):


,,world_production_t,world_area_ha,world_yield_t_per_ha
Item Code,Year,,,
1717,2023,"3,768,503,439.2700","839,332,298.0000",4.4899
1720,2023,"1,085,273,164.8100","78,152,798.0000",13.8866
1726,2023,"98,505,370.4300","92,205,777.0000",1.0683
1732,2023,"263,010,876.8900","358,416,770.0000",0.7338
1735,2023,"1,781,927,135.9900","83,139,439.0000",21.4330
1738,2023,"1,218,679,287.5300","83,799,849.0000",14.5427


In [7]:
# Step 3: Compute ha-equivalents for each crop
usa_prod_merged = usa_prod.merge(
    world_crop_yield[["world_yield_t_per_ha"]].reset_index(),
    on=["Item Code", "Year"],
    how="left",
)
usa_prod_merged["ha_equivalent"] = np.where(
    usa_prod_merged["world_yield_t_per_ha"] > 0,
    usa_prod_merged["Value"] / usa_prod_merged["world_yield_t_per_ha"],
    0,
)

print("Top 10 crops: production, world yield, ha-equivalents:")
display(
    usa_prod_merged[["Item", "Value", "world_yield_t_per_ha", "ha_equivalent"]]
    .head(10)
    .rename(columns={"Value": "production_t"})
)

Top 10 crops: production, world yield, ha-equivalents:


,Item,production_t,world_yield_t_per_ha,ha_equivalent
0,"Cereals, primary","462,631,157.5500",4.4899,"103,038,587.8242"
1,Vegetables Primary,"31,253,511.2700",21.4330,"1,458,196.2086"
2,"Oilcrops, Oil Equivalent","23,240,551.3100",0.7338,"31,670,946.2059"
3,Fruit Primary,"21,876,188.6900",14.5427,"1,504,268.8652"
4,"Roots and Tubers, Total","21,131,470.4800",13.8866,"1,521,721.5328"
5,"Pulses, Total","2,367,610.0000",1.0683,"2,216,197.1345"


In [8]:
# Step 4: Sum and multiply by EQF
total_ha = usa_prod_merged["ha_equivalent"].sum()
ef_cropland = total_ha * EQF_CROPLAND

stored_val = stored(DEMO_COUNTRY, DEMO_YEAR, "ef_cropland_gha")

print(f"Total ha-equivalents across all {len(usa_prod_merged)} crops: {total_ha:,.0f} ha")
print(f"EF_cropland = {total_ha:,.0f} x {EQF_CROPLAND} = {ef_cropland:,.0f} gha")
print(f"\nStored result:  {stored_val:,.0f} gha")
print(f"Computed here:  {ef_cropland:,.0f} gha")
print(f"Match: {abs(ef_cropland - stored_val) / stored_val < 0.001}")

Total ha-equivalents across all 6 crops: 141,409,918 ha
EF_cropland = 141,409,918 x 2.1 = 296,960,827 gha

Stored result:  296,960,827 gha
Computed here:  296,960,827 gha
Match: True


---
## Section 4: Cropland BC \u2014 Step by Step

**Formula:**
$$\text{BC}_{\text{cropland}} = \text{CroplandArea}_{ha} \times \text{YF} \times \text{EQF}_{\text{cropland}} (2.1)$$

Where:
- CroplandArea = Arable land (Item 6621) + Permanent crops (Item 6650), in 1000 ha \u2192 convert to ha
- YF (Yield Factor) = national average yield / world average yield
- YF is clipped to [0.01, 10.0] to prevent outliers

In [9]:
# Step 1: Get cropland area for USA 2023
usa_cla = cropland_area[
    (cropland_area["Area Code"] == DEMO_COUNTRY)
    & (cropland_area["Year"] == DEMO_YEAR)
    & (cropland_area["Item Code"].isin([6621, 6650]))
]
print("USA 2023 cropland area:")
display(usa_cla[["Item Code", "Item", "Value", "Unit"]])

area_1000ha = usa_cla["Value"].sum()
area_ha = area_1000ha * 1000
print(f"\nTotal cropland: {area_1000ha:,.1f} thousand ha = {area_ha:,.0f} ha")

USA 2023 cropland area:


,Item Code,Item,Value,Unit
36714,6621,Arable land,"151,563.5251",1000 ha
36777,6650,Permanent crops,"3,170.6000",1000 ha



Total cropland: 154,734.1 thousand ha = 154,734,125 ha


In [10]:
# Step 2: Compute national yield (aggregate across all crops)
nat_prod_rows = crop_prod[
    (crop_prod["Area Code"] == DEMO_COUNTRY)
    & (crop_prod["Year"] == DEMO_YEAR)
    & (crop_prod["Element"] == "Production")
]
nat_area_rows = crop_prod[
    (crop_prod["Area Code"] == DEMO_COUNTRY)
    & (crop_prod["Year"] == DEMO_YEAR)
    & (crop_prod["Element"] == "Area harvested")
]
nat_prod_total = nat_prod_rows["Value"].sum()
nat_area_total = nat_area_rows["Value"].sum()
nat_yield = nat_prod_total / nat_area_total

print(f"USA national production total: {nat_prod_total:,.0f} t")
print(f"USA national harvested area:   {nat_area_total:,.0f} ha")
print(f"USA national yield:            {nat_yield:,.4f} t/ha")

USA national production total: 562,500,489 t
USA national harvested area:   97,530,289 ha
USA national yield:            5.7674 t/ha


In [11]:
# Step 3: Compute world yield (aggregate across all crops, all countries)
wld = world_crop_yield.loc[world_crop_yield.index.get_level_values("Year") == DEMO_YEAR]
world_prod_total = wld["world_production_t"].sum()
world_area_total = wld["world_area_ha"].sum()
world_yield = world_prod_total / world_area_total

print(f"World production total: {world_prod_total:,.0f} t")
print(f"World harvested area:   {world_area_total:,.0f} ha")
print(f"World average yield:    {world_yield:,.4f} t/ha")

World production total: 8,215,899,275 t
World harvested area:   1,535,046,931 ha
World average yield:    5.3522 t/ha


In [12]:
# Step 4: YF and final BC
yf = nat_yield / world_yield
yf_clipped = np.clip(yf, 0.01, 10.0)

bc_cropland = area_ha * yf_clipped * EQF_CROPLAND
stored_val = stored(DEMO_COUNTRY, DEMO_YEAR, "bc_cropland_gha")

print(f"Yield Factor (YF) = {nat_yield:,.4f} / {world_yield:,.4f} = {yf:,.4f}")
print(f"YF clipped [0.01, 10.0]: {yf_clipped:,.4f}")
print(f"\nBC_cropland = {area_ha:,.0f} x {yf_clipped:,.4f} x {EQF_CROPLAND} = {bc_cropland:,.0f} gha")
print(f"\nStored result:  {stored_val:,.0f} gha")
print(f"Computed here:  {bc_cropland:,.0f} gha")
print(f"Match: {abs(bc_cropland - stored_val) / stored_val < 0.001}")

Yield Factor (YF) = 5.7674 / 5.3522 = 1.0776
YF clipped [0.01, 10.0]: 1.0776

BC_cropland = 154,734,125 x 1.0776 x 2.1 = 350,150,978 gha

Stored result:  350,150,978 gha
Computed here:  350,150,978 gha
Match: True


---
## Section 5: Grazing EF \u2014 Step by Step

**Formula:**
$$\text{EF}_{\text{grazing}} = \frac{\sum (\text{Heads} \times \text{PastureDM}_{t/head})}{\text{NPP}_{\text{grassland}} (2.45)} \times \text{EQF}_{\text{grazing}} (0.5)$$

Where:
- Heads = livestock stock count from `livestock_stocks.csv`
- PastureDM = annual pasture dry matter demand per head from GLEAM coefficients
- NPP = grassland net primary productivity (2.45 t DM/ha/yr)

In [13]:
# Step 1: USA 2023 livestock stocks
usa_ls = livestock[
    (livestock["Area Code"] == DEMO_COUNTRY) & (livestock["Year"] == DEMO_YEAR)
].copy()
print("USA 2023 livestock stocks:")
display(usa_ls[["Item", "Value", "Unit"]])

USA 2023 livestock stocks:


,Item,Value,Unit
47629,Cattle,"88,841,000.0000",An
47693,Goats,"2,527,000.0000",An
47757,Sheep,"5,130,000.0000",An
47821,Swine / pigs,"75,461,300.0000",An


In [14]:
# Step 2: GLEAM pasture demand mapping
gleam_table = pd.DataFrame(
    list(GLEAM_PASTURE.items()),
    columns=["FAOSTAT Item", "Pasture DM (t/head/yr)"]
)
print("GLEAM pasture demand mapping (as used in code):")
display(gleam_table)

GLEAM pasture demand mapping (as used in code):


,FAOSTAT Item,Pasture DM (t/head/yr)
0,Cattle,2.3500
1,Buffalo,2.4090
2,Sheep,0.4600
3,Goats,0.3940
4,Camels,2.0000
5,Swine / pigs,0.0000


In [15]:
# Step 3: Compute pasture demand per animal type
usa_ls["pasture_demand_per_head"] = usa_ls["Item"].map(GLEAM_PASTURE).fillna(0)
usa_ls["total_pasture_demand_t"] = usa_ls["Value"].fillna(0) * usa_ls["pasture_demand_per_head"]

print("Pasture demand by livestock type:")
display(
    usa_ls[["Item", "Value", "pasture_demand_per_head", "total_pasture_demand_t"]]
    .rename(columns={"Value": "heads"})
)

total_demand = usa_ls["total_pasture_demand_t"].sum()
print(f"\nTotal pasture demand: {total_demand:,.0f} tonnes DM")

Pasture demand by livestock type:


,Item,heads,pasture_demand_per_head,total_pasture_demand_t
47629,Cattle,"88,841,000.0000",2.3500,"208,776,350.0000"
47693,Goats,"2,527,000.0000",0.3940,"995,638.0000"
47757,Sheep,"5,130,000.0000",0.4600,"2,359,800.0000"
47821,Swine / pigs,"75,461,300.0000",0.0000,0.0000



Total pasture demand: 212,131,788 tonnes DM


In [16]:
# Step 4: Divide by NPP, multiply by EQF
grazing_ha = total_demand / GRASSLAND_NPP
ef_grazing = grazing_ha * EQF_GRAZING
stored_val = stored(DEMO_COUNTRY, DEMO_YEAR, "ef_grazing_gha")

print(f"Grazing area = {total_demand:,.0f} / {GRASSLAND_NPP} = {grazing_ha:,.0f} ha")
print(f"EF_grazing = {grazing_ha:,.0f} x {EQF_GRAZING} = {ef_grazing:,.0f} gha")
print(f"\nStored result:  {stored_val:,.0f} gha")
print(f"Computed here:  {ef_grazing:,.0f} gha")
print(f"Match: {abs(ef_grazing - stored_val) / stored_val < 0.001}")

Grazing area = 212,131,788 / 2.45 = 86,584,403 ha
EF_grazing = 86,584,403 x 0.5 = 43,292,202 gha

Stored result:  43,292,202 gha
Computed here:  43,292,202 gha
Match: True


---
## Section 6: Grazing BC \u2014 Step by Step

**Formula:**
$$\text{BC}_{\text{grazing}} = \text{PastureArea}_{ha} \times \text{YF} (1.0) \times \text{EQF}_{\text{grazing}} (0.5)$$

Pasture area is in 1000 ha. YF for grazing is set to 1.0 (no country-specific adjustment).

In [17]:
# USA 2023 pasture area
usa_pa = pasture_area[
    (pasture_area["Area Code"] == DEMO_COUNTRY) & (pasture_area["Year"] == DEMO_YEAR)
]
print("USA 2023 pasture area:")
display(usa_pa[["Item", "Value", "Unit"]])

pa_1000ha = usa_pa["Value"].sum()
pa_ha = pa_1000ha * 1000
bc_grazing = pa_ha * 1.0 * EQF_GRAZING
stored_val = stored(DEMO_COUNTRY, DEMO_YEAR, "bc_grazing_gha")

print(f"\nPasture area: {pa_1000ha:,.1f} thousand ha = {pa_ha:,.0f} ha")
print(f"BC_grazing = {pa_ha:,.0f} x 1.0 x {EQF_GRAZING} = {bc_grazing:,.0f} gha")
print(f"\nStored result:  {stored_val:,.0f} gha")
print(f"Computed here:  {bc_grazing:,.0f} gha")
print(f"Match: {abs(bc_grazing - stored_val) / stored_val < 0.001}")

USA 2023 pasture area:


,Item,Value,Unit
11724,Permanent meadows and pastures,"266,806.2423",1000 ha



Pasture area: 266,806.2 thousand ha = 266,806,242 ha
BC_grazing = 266,806,242 x 1.0 x 0.5 = 133,403,121 gha

Stored result:  133,403,121 gha
Computed here:  133,403,121 gha
Match: True


---
## Section 7: Forest EF \u2014 Step by Step

**Formula:**
$$\text{EF}_{\text{forest}} = \frac{\text{Roundwood}_{m^3}}{\text{NAI} (1.81)} \times \text{EQF}_{\text{forest}} (1.3)$$

Where:
- Roundwood = national roundwood production in m\u00b3 from `timber_production.csv`
- NAI = Net Annual Increment (world-average forest growth rate)

In [18]:
# USA 2023 roundwood production
usa_timber = timber[
    (timber["Area Code"] == DEMO_COUNTRY)
    & (timber["Year"] == DEMO_YEAR)
    & (timber["Item"] == "Roundwood")
    & (timber["Element"] == "Production")
]
print("USA 2023 timber production:")
display(usa_timber[["Item", "Element", "Value", "Unit"]])

roundwood_m3 = usa_timber["Value"].sum()
print(f"\nRoundwood production: {roundwood_m3:,.0f} m\u00b3")

USA 2023 timber production:


,Item,Element,Value,Unit
95804,Roundwood,Production,"389,026,299.0000",m3



Roundwood production: 389,026,299 m³


In [19]:
# Compute EF
forest_ha = roundwood_m3 / FOREST_NAI
ef_forest = forest_ha * EQF_FOREST
stored_val = stored(DEMO_COUNTRY, DEMO_YEAR, "ef_forest_gha")

print(f"Forest area needed = {roundwood_m3:,.0f} / {FOREST_NAI} = {forest_ha:,.0f} ha")
print(f"EF_forest = {forest_ha:,.0f} x {EQF_FOREST} = {ef_forest:,.0f} gha")
print(f"\nStored result:  {stored_val:,.0f} gha")
print(f"Computed here:  {ef_forest:,.0f} gha")
print(f"Match: {abs(ef_forest - stored_val) / stored_val < 0.001}")

Forest area needed = 389,026,299 / 1.81 = 214,931,657 ha
EF_forest = 214,931,657 x 1.3 = 279,411,154 gha

Stored result:  279,411,154 gha
Computed here:  279,411,154 gha
Match: True


---
## Section 8: Forest BC \u2014 Step by Step

**Formula:**
$$\text{BC}_{\text{forest}} = \text{ForestArea}_{ha} \times \text{YF} (1.0) \times \text{EQF}_{\text{forest}} (1.3)$$

Forest area is in 1000 ha. YF for forest is set to 1.0.

In [20]:
# USA 2023 forest area
usa_fa = forest_area[
    (forest_area["Area Code"] == DEMO_COUNTRY) & (forest_area["Year"] == DEMO_YEAR)
]
print("USA 2023 forest area:")
display(usa_fa[["Item", "Value", "Unit"]])

fa_1000ha = usa_fa["Value"].sum()
fa_ha = fa_1000ha * 1000
bc_forest = fa_ha * 1.0 * EQF_FOREST
stored_val = stored(DEMO_COUNTRY, DEMO_YEAR, "bc_forest_gha")

print(f"\nForest area: {fa_1000ha:,.1f} thousand ha = {fa_ha:,.0f} ha")
print(f"BC_forest = {fa_ha:,.0f} x 1.0 x {EQF_FOREST} = {bc_forest:,.0f} gha")
print(f"\nStored result:  {stored_val:,.0f} gha")
print(f"Computed here:  {bc_forest:,.0f} gha")
print(f"Match: {abs(bc_forest - stored_val) / stored_val < 0.001}")

USA 2023 forest area:


,Item,Value,Unit
14723,Forest land,"309,039.0000",1000 ha
14759,Forest land,33.7800,%



Forest area: 309,072.8 thousand ha = 309,072,780 ha
BC_forest = 309,072,780 x 1.0 x 1.3 = 401,794,614 gha

Stored result:  401,794,614 gha
Computed here:  401,794,614 gha
Match: True


---
## Section 9: Fishing EF \u2014 Step by Step

**Formula (PPR method):**

For each fish category:
$$\text{PPR} = \text{prod}_t \times \text{DR} (1.27) \times \left(\frac{1}{\text{TE} (0.1013)}\right)^{\text{TL}-1} \div \text{WC} (9.0)$$

Then:
$$\text{marine\_yield} = \frac{\text{World\_Sustainable\_PPR}}{\text{Continental\_Shelf\_Area} (2 \times 10^9 \text{ ha})}$$
$$\text{EF}_{\text{fishing}} = \frac{\sum \text{PPR}}{\text{marine\_yield}} \times \text{EQF}_{\text{fishing}} (0.4)$$

Where:
- DR = Discard Rate (accounts for bycatch)
- TE = Transfer Efficiency between trophic levels
- TL = Trophic Level of each fish category
- WC = wet-weight to carbon conversion ratio
- World sustainable PPR computed from 93 Mt sustainable catch at average TL 3.1

In [21]:
# Step 1: USA 2023 fish production from FBS
usa_fish = fish_fbs[
    (fish_fbs["Area Code"] == DEMO_COUNTRY)
    & (fish_fbs["Year"] == DEMO_YEAR)
    & (fish_fbs["Element"] == "Production")
].copy()
print(f"USA 2023 fish production ({len(usa_fish)} items):")
display(usa_fish[["Item", "Value", "Unit"]])

USA 2023 fish production (13 items):


,Item,Value,Unit
136332,"Fish, Body Oil",85.0000,1000 t
147915,"Fish, Liver Oil",1.0000,1000 t
251428,"Fish, Seafood","5,291.0000",1000 t
273431,Freshwater Fish,601.0000,1000 t
292826,Demersal Fish,"2,559.0000",1000 t
310135,Pelagic Fish,838.0000,1000 t
324724,"Marine Fish, Other",330.0000,1000 t
335846,Crustaceans,257.0000,1000 t
344368,Cephalopods,54.0000,1000 t
351765,"Molluscs, Other",646.0000,1000 t


In [22]:
# Step 2: Trophic level mapping
tl_table = pd.DataFrame(
    list(TROPHIC_LEVELS.items()),
    columns=["FBS Fish Category", "Trophic Level"]
)
print("Trophic level mapping:")
display(tl_table)

Trophic level mapping:


,FBS Fish Category,Trophic Level
0,Freshwater Fish,2.5000
1,Demersal Fish,3.8000
2,Pelagic Fish,3.0000
3,"Marine Fish, Other",3.2000
4,Crustaceans,2.5000
5,Cephalopods,3.5000
6,"Molluscs, Other",2.1000
7,"Aquatic Animals, Others",3.0000
8,"Aquatic Products, Other",3.0000
9,"Meat, Aquatic Mammals",3.2000


In [23]:
# Step 3: Compute PPR for each fish category
usa_fish["tl"] = usa_fish["Item"].map(TROPHIC_LEVELS)
usa_fish_valid = usa_fish.dropna(subset=["tl"]).copy()
usa_fish_valid["prod_t"] = usa_fish_valid["Value"].fillna(0) * 1000  # 1000 t -> t
usa_fish_valid["ppr"] = (
    usa_fish_valid["prod_t"]
    * DISCARD_RATE
    * (1.0 / TRANSFER_EFFICIENCY) ** (usa_fish_valid["tl"] - 1)
    / WET_TO_CARBON
)

print("PPR calculation per fish category:")
display(
    usa_fish_valid[["Item", "Value", "prod_t", "tl", "ppr"]]
    .rename(columns={"Value": "prod_kt"})
)

total_ppr = usa_fish_valid["ppr"].sum()
print(f"\nTotal PPR: {total_ppr:,.0f} t C")

PPR calculation per fish category:


,Item,prod_kt,prod_t,tl,ppr
273431,Freshwater Fish,601.0000,"601,000.0000",2.5000,"2,630,398.3018"
292826,Demersal Fish,"2,559.0000","2,559,000.0000",3.8000,"219,748,060.3596"
310135,Pelagic Fish,838.0000,"838,000.0000",3.0000,"11,523,551.2972"
324724,"Marine Fish, Other",330.0000,"330,000.0000",3.2000,"7,173,553.9955"
335846,Crustaceans,257.0000,"257,000.0000",2.5000,"1,124,812.5850"
344368,Cephalopods,54.0000,"54,000.0000",3.5000,"2,333,089.2786"
351765,"Molluscs, Other",646.0000,"646,000.0000",2.1000,"1,131,418.6666"
357964,"Aquatic Animals, Others",5.0000,"5,000.0000",3.0000,"68,756.2727"
363372,"Aquatic Products, Other",3.0000,"3,000.0000",3.0000,"41,253.7636"



Total PPR: 245,774,895 t C


In [24]:
# Step 4: Compute marine yield and EF
avg_tl = 3.1
world_sust_ppr = (
    SUSTAINABLE_CATCH_MT * 1e6
    * DISCARD_RATE
    * (1.0 / TRANSFER_EFFICIENCY) ** (avg_tl - 1)
    / WET_TO_CARBON
)
marine_yield = world_sust_ppr / CONTINENTAL_SHELF_HA

fishing_area_ha = total_ppr / marine_yield
ef_fishing = fishing_area_ha * EQF_FISHING
stored_val = stored(DEMO_COUNTRY, DEMO_YEAR, "ef_fishing_gha")

print(f"World sustainable PPR: {world_sust_ppr:,.0f} t C")
print(f"Marine yield = {world_sust_ppr:,.0f} / {CONTINENTAL_SHELF_HA:,.0f} = {marine_yield:.6f} t C/ha")
print(f"Fishing area = {total_ppr:,.0f} / {marine_yield:.6f} = {fishing_area_ha:,.0f} ha")
print(f"EF_fishing = {fishing_area_ha:,.0f} x {EQF_FISHING} = {ef_fishing:,.0f} gha")
print(f"\nStored result:  {stored_val:,.0f} gha")
print(f"Computed here:  {ef_fishing:,.0f} gha")
print(f"Match: {abs(ef_fishing - stored_val) / stored_val < 0.001}")

World sustainable PPR: 1,607,919,584 t C
Marine yield = 1,607,919,584 / 2,000,000,000 = 0.803960 t C/ha
Fishing area = 245,774,895 / 0.803960 = 305,705,456 ha
EF_fishing = 305,705,456 x 0.4 = 122,282,182 gha

Stored result:  122,282,182 gha
Computed here:  122,282,182 gha
Match: True


---
## Section 10: Fishing BC \u2014 Step by Step

**Formula:**
$$\text{BC}_{\text{fishing}} = \text{WorldFishingBC} \times \frac{\text{CountryProd}}{\text{WorldProd}}$$

Where:
- WorldFishingBC = Continental Shelf Area (2 billion ha) \u00d7 EQF_fishing (0.4) = 800 million gha
- Production share is computed from FBS fish production data (only categories with trophic levels)

In [25]:
# World fishing BC
world_fishing_bc = CONTINENTAL_SHELF_HA * EQF_FISHING
print(f"World Fishing BC = {CONTINENTAL_SHELF_HA:,.0f} ha x {EQF_FISHING} = {world_fishing_bc:,.0f} gha")
print(f"                 = {world_fishing_bc / 1e6:,.0f} million gha")

# Country production share (using trophic-level-valid items only, matching the script)
_ff_all = fish_fbs[
    (fish_fbs["Area Code"] < 5000)
    & (fish_fbs["Element"] == "Production")
].copy()
_ff_all["tl"] = _ff_all["Item"].map(TROPHIC_LEVELS)
_ff_all = _ff_all.dropna(subset=["tl"])
_ff_all["prod_t"] = _ff_all["Value"].fillna(0) * 1000

world_prod_t = _ff_all[_ff_all["Year"] == DEMO_YEAR]["prod_t"].sum()
country_prod_t = usa_fish_valid["prod_t"].sum()
share = country_prod_t / world_prod_t

print(f"\nUSA fish production:   {country_prod_t:,.0f} t")
print(f"World fish production:  {world_prod_t:,.0f} t")
print(f"USA share: {share:.6f} ({share * 100:.2f}%)")

World Fishing BC = 2,000,000,000 ha x 0.4 = 800,000,000 gha
                 = 800 million gha

USA fish production:   5,293,000 t
World fish production:  299,110,000 t
USA share: 0.017696 (1.77%)


In [26]:
# Final BC
bc_fishing = world_fishing_bc * share
stored_val = stored(DEMO_COUNTRY, DEMO_YEAR, "bc_fishing_gha")

print(f"BC_fishing = {world_fishing_bc:,.0f} x {share:.6f} = {bc_fishing:,.0f} gha")
print(f"\nStored result:  {stored_val:,.0f} gha")
print(f"Computed here:  {bc_fishing:,.0f} gha")
print(f"Match: {abs(bc_fishing - stored_val) / stored_val < 0.001}")

BC_fishing = 800,000,000 x 0.017696 = 14,156,665 gha

Stored result:  14,156,665 gha
Computed here:  14,156,665 gha
Match: True


---
## Section 11: Built-up EF/BC \u2014 Step by Step

**Formula:**
$$\text{EF}_{\text{built\text{-}up}} = \text{BC}_{\text{built\text{-}up}} = \text{BuiltUpArea}_{ha} \times \text{EQF}_{\text{built\text{-}up}} (2.2)$$

**3-tier estimation for built-up area:**
1. Use Item 6649 (\"Farm buildings & Farmyards\") directly if available
2. Else compute residual: Land area (6601) \u2212 Agricultural land (6610) \u2212 Forest (6646) \u2212 Other land (6670) \u2212 Inland water (6680); use if positive
3. Else fallback: 3% of cropland area

In [27]:
# Step 1: Show all relevant land-use items for USA 2023
items_of_interest = [6601, 6610, 6646, 6649, 6670, 6680]
usa_lu = land_use_all[
    (land_use_all["Area Code"] == DEMO_COUNTRY)
    & (land_use_all["Year"] == DEMO_YEAR)
    & (land_use_all["Item Code"].isin(items_of_interest))
]
print("USA 2023 land-use items:")
display(usa_lu[["Item Code", "Item", "Value", "Unit"]].sort_values("Item Code"))

USA 2023 land-use items:


,Item Code,Item,Value,Unit
332307,6601,Land area,"914,742.0000",1000 ha
332433,6610,Agricultural land,"421,540.3674",1000 ha
332496,6610,Agricultural land,46.0800,%
332559,6610,Agricultural land,895.4600,USD_PPP/ha
333392,6646,Forest land,"309,039.0000",1000 ha
333428,6646,Forest land,"22,800.7440",million t
333464,6646,Forest land,33.7800,%
333358,6649,Farm buildings and Farmyards,"13,231.6592",1000 ha
333678,6670,Other land,"170,930.9734",1000 ha
333741,6680,Inland waters,"68,409.0000",1000 ha


In [28]:
# Step 2: Compute residual and determine built-up area
items_needed = {6601: 0, 6610: 0, 6646: 0, 6670: 0, 6680: 0}
for _, r in usa_lu.iterrows():
    if r["Item Code"] in items_needed:
        items_needed[r["Item Code"]] = r["Value"] if not pd.isna(r["Value"]) else 0

land_area = items_needed[6601]
ag_land = items_needed[6610]
forest_land = items_needed[6646]
other_land = items_needed[6670]
inland_water = items_needed[6680]
residual = land_area - ag_land - forest_land - other_land - inland_water

# Check if 6649 is directly available
direct_6649 = usa_lu[usa_lu["Item Code"] == 6649]["Value"]
has_6649 = len(direct_6649) > 0 and not pd.isna(direct_6649.iloc[0]) if len(direct_6649) > 0 else False

print(f"Land area (6601):       {land_area:>10,.1f} thousand ha")
print(f"Agricultural land (6610): {ag_land:>10,.1f} thousand ha")
print(f"Forest land (6646):     {forest_land:>10,.1f} thousand ha")
print(f"Other land (6670):      {other_land:>10,.1f} thousand ha")
print(f"Inland water (6680):    {inland_water:>10,.1f} thousand ha")
print(f"Residual:               {residual:>10,.1f} thousand ha")
print(f"\nItem 6649 available: {has_6649}")

if has_6649:
    built_up_1000ha = direct_6649.iloc[0]
    print(f"Using item 6649: {built_up_1000ha:,.1f} thousand ha")
elif residual > 0:
    built_up_1000ha = residual
    print(f"Using positive residual: {built_up_1000ha:,.1f} thousand ha")
else:
    built_up_1000ha = (area_1000ha) * 0.03  # cropland area from Section 4
    print(f"Fallback: 3% of cropland = {built_up_1000ha:,.1f} thousand ha")

Land area (6601):        914,742.0 thousand ha
Agricultural land (6610):      895.5 thousand ha
Forest land (6646):           33.8 thousand ha
Other land (6670):       170,931.0 thousand ha
Inland water (6680):      68,409.0 thousand ha
Residual:                674,472.8 thousand ha

Item 6649 available: True
Using item 6649: 13,231.7 thousand ha


In [29]:
# Step 3: Compute EF = BC = built_up_ha x EQF
built_up_ha = built_up_1000ha * 1000
ef_built_up = built_up_ha * EQF_BUILT_UP
bc_built_up = ef_built_up  # EF = BC for built-up
stored_ef = stored(DEMO_COUNTRY, DEMO_YEAR, "ef_built_up_gha")
stored_bc = stored(DEMO_COUNTRY, DEMO_YEAR, "bc_built_up_gha")

print(f"Built-up area: {built_up_1000ha:,.1f} thousand ha = {built_up_ha:,.0f} ha")
print(f"EF_built_up = BC_built_up = {built_up_ha:,.0f} x {EQF_BUILT_UP} = {ef_built_up:,.0f} gha")
print(f"\nStored EF: {stored_ef:,.0f} gha  |  Computed: {ef_built_up:,.0f} gha  |  Match: {abs(ef_built_up - stored_ef) / stored_ef < 0.01}")
print(f"Stored BC: {stored_bc:,.0f} gha  |  Computed: {bc_built_up:,.0f} gha  |  Match: {abs(bc_built_up - stored_bc) / stored_bc < 0.01}")

Built-up area: 13,231.7 thousand ha = 13,231,659 ha
EF_built_up = BC_built_up = 13,231,659 x 2.2 = 29,109,650 gha

Stored EF: 29,109,650 gha  |  Computed: 29,109,650 gha  |  Match: True
Stored BC: 29,109,650 gha  |  Computed: 29,109,650 gha  |  Match: True


---
## Section 12: Carbon EF \u2014 Step by Step

**Formula:**
$$\text{EF}_{\text{carbon}} = \frac{\text{CO}_2\text{kt} \times 1000 \times (1 - 0.35)}{\text{AFCS} (0.73) \times \frac{44}{12}} \times \text{EQF}_{\text{carbon}} (1.3)$$

Where:
- CO\u2082 emissions from energy use, in kilotonnes \u2192 convert to tonnes
- 35% absorbed by oceans \u2192 65% must be sequestered by forests
- AFCS = 0.73 t C/ha/yr \u2192 convert to t CO\u2082/ha/yr: 0.73 \u00d7 (44/12) = 2.677
- Then multiply by forest EQF

In [30]:
# Step 1: USA 2023 CO2 emissions from energy
usa_co2 = co2_energy[
    (co2_energy["Area Code"] == DEMO_COUNTRY) & (co2_energy["Year"] == DEMO_YEAR)
]
print("USA 2023 CO2 emissions from energy:")
display(usa_co2[["Item", "Element", "Value", "Unit"]])

co2_kt = usa_co2["Value"].sum()
print(f"\nTotal CO2: {co2_kt:,.2f} kt")

USA 2023 CO2 emissions from energy:


,Item,Element,Value,Unit
11612,Energy,Emissions (CO2),"4,640,000.0000",kt



Total CO2: 4,640,000.00 kt


In [31]:
# Step 2: Convert through the formula step-by-step
co2_tonnes = co2_kt * 1000
print(f"Step 1: kt to tonnes:     {co2_kt:,.2f} x 1000 = {co2_tonnes:,.0f} t CO2")

land_co2 = co2_tonnes * (1 - OCEAN_CO2_FRAC)
print(f"Step 2: Land fraction:    {co2_tonnes:,.0f} x (1 - {OCEAN_CO2_FRAC}) = {land_co2:,.0f} t CO2")

seq_rate_co2 = AFCS * (44.0 / 12.0)
print(f"Step 3: Seq rate (CO2):   {AFCS} x (44/12) = {seq_rate_co2:,.4f} t CO2/ha/yr")

carbon_area_ha = land_co2 / seq_rate_co2
print(f"Step 4: Area needed:      {land_co2:,.0f} / {seq_rate_co2:,.4f} = {carbon_area_ha:,.0f} ha")

ef_carbon = carbon_area_ha * EQF_CARBON
print(f"Step 5: Apply EQF:        {carbon_area_ha:,.0f} x {EQF_CARBON} = {ef_carbon:,.0f} gha")

stored_val = stored(DEMO_COUNTRY, DEMO_YEAR, "ef_carbon_gha")
print(f"\nStored result:  {stored_val:,.0f} gha")
print(f"Computed here:  {ef_carbon:,.0f} gha")
print(f"Match: {abs(ef_carbon - stored_val) / stored_val < 0.001}")

Step 1: kt to tonnes:     4,640,000.00 x 1000 = 4,640,000,000 t CO2
Step 2: Land fraction:    4,640,000,000 x (1 - 0.35) = 3,016,000,000 t CO2
Step 3: Seq rate (CO2):   0.73 x (44/12) = 2.6767 t CO2/ha/yr
Step 4: Area needed:      3,016,000,000 / 2.6767 = 1,126,774,595 ha
Step 5: Apply EQF:        1,126,774,595 x 1.3 = 1,464,806,974 gha

Stored result:  1,464,806,974 gha
Computed here:  1,464,806,974 gha
Match: True


---
## Section 13: Assembly & Final Results

All 6 EF and 5 BC components assembled into totals.

$$\text{EF}_{\text{total}} = \text{EF}_{\text{cropland}} + \text{EF}_{\text{grazing}} + \text{EF}_{\text{forest}} + \text{EF}_{\text{fishing}} + \text{EF}_{\text{built\text{-}up}} + \text{EF}_{\text{carbon}}$$

$$\text{BC}_{\text{total}} = \text{BC}_{\text{cropland}} + \text{BC}_{\text{grazing}} + \text{BC}_{\text{forest}} + \text{BC}_{\text{fishing}} + \text{BC}_{\text{built\text{-}up}}$$

$$\text{Ecological Deficit} = \text{BC}_{\text{total}} - \text{EF}_{\text{total}}$$

In [32]:
# Assemble all components for USA 2023
component_data = pd.DataFrame({
    "Component": ["Cropland", "Grazing", "Forest", "Fishing", "Built-up", "Carbon"],
    "EF (gha)": [ef_cropland, ef_grazing, ef_forest, ef_fishing, ef_built_up, ef_carbon],
    "BC (gha)": [bc_cropland, bc_grazing, bc_forest, bc_fishing, bc_built_up, 0],
})
component_data["EF (gha)"] = component_data["EF (gha)"].round(0)
component_data["BC (gha)"] = component_data["BC (gha)"].round(0)

ef_total = component_data["EF (gha)"].sum()
bc_total = component_data["BC (gha)"].sum()
deficit = bc_total - ef_total

# Add totals row
totals = pd.DataFrame({
    "Component": ["TOTAL"],
    "EF (gha)": [ef_total],
    "BC (gha)": [bc_total],
})
summary = pd.concat([component_data, totals], ignore_index=True)

print("USA 2023 \u2014 All Components:")
display(summary.style.format({"EF (gha)": "{:,.0f}", "BC (gha)": "{:,.0f}"}))

print(f"\nEcological Deficit: {deficit:,.0f} gha")

USA 2023 — All Components:


,Component,EF (gha),BC (gha)
0,Cropland,"296,960,827","350,150,978"
1,Grazing,"43,292,202","133,403,121"
2,Forest,"279,411,154","401,794,614"
3,Fishing,"122,282,182","14,156,665"
4,Built-up,"29,109,650","29,109,650"
5,Carbon,"1,464,806,974",0
6,TOTAL,"2,235,862,989","928,615,028"



Ecological Deficit: -1,307,247,961 gha


In [33]:
# Per capita
usa_pop_row = population[
    (population["Area Code"] == DEMO_COUNTRY)
    & (population["Year"] == DEMO_YEAR)
    & (population["Element"] == "Total Population - Both sexes")
]
pop_thousands = usa_pop_row["Value"].iloc[0]
pop_actual = pop_thousands * 1000

ef_per_cap = ef_total / pop_actual
bc_per_cap = bc_total / pop_actual

stored_ef_total = stored(DEMO_COUNTRY, DEMO_YEAR, "ef_total_gha")
stored_bc_total = stored(DEMO_COUNTRY, DEMO_YEAR, "bc_total_gha")
stored_ef_pc = stored(DEMO_COUNTRY, DEMO_YEAR, "ef_per_capita_gha")
stored_bc_pc = stored(DEMO_COUNTRY, DEMO_YEAR, "bc_per_capita_gha")

print(f"USA population: {pop_thousands:,.1f} thousand = {pop_actual:,.0f} people")
print(f"\nEF total:  computed {ef_total:,.0f}  |  stored {stored_ef_total:,.0f}")
print(f"BC total:  computed {bc_total:,.0f}  |  stored {stored_bc_total:,.0f}")
print(f"EF/cap:    computed {ef_per_cap:,.4f}  |  stored {stored_ef_pc:,.4f}")
print(f"BC/cap:    computed {bc_per_cap:,.4f}  |  stored {stored_bc_pc:,.4f}")

USA population: 343,477.3 thousand = 343,477,335 people

EF total:  computed 2,235,862,989  |  stored 2,235,862,989
BC total:  computed 928,615,028  |  stored 928,615,028
EF/cap:    computed 6.5095  |  stored 6.5095
BC/cap:    computed 2.7036  |  stored 2.7036


In [34]:
# Compare with stored values for validation
print("Detailed comparison with stored results:")
cols = [
    "ef_cropland_gha", "ef_grazing_gha", "ef_forest_gha",
    "ef_fishing_gha", "ef_built_up_gha", "ef_carbon_gha",
    "ef_total_gha",
    "bc_cropland_gha", "bc_grazing_gha", "bc_forest_gha",
    "bc_fishing_gha", "bc_built_up_gha", "bc_total_gha",
]
computed = [
    ef_cropland, ef_grazing, ef_forest, ef_fishing, ef_built_up, ef_carbon,
    ef_total,
    bc_cropland, bc_grazing, bc_forest, bc_fishing, bc_built_up, bc_total,
]
stored_vals = [stored(DEMO_COUNTRY, DEMO_YEAR, c) for c in cols]

comparison = pd.DataFrame({
    "Component": cols,
    "Computed": computed,
    "Stored": stored_vals,
})
comparison["Diff (%)"] = (
    (comparison["Computed"] - comparison["Stored"]) / comparison["Stored"].replace(0, np.nan) * 100
)
display(comparison.style.format({"Computed": "{:,.0f}", "Stored": "{:,.0f}", "Diff (%)": "{:+.3f}%"}))

Detailed comparison with stored results:


,Component,Computed,Stored,Diff (%)
0,ef_cropland_gha,"296,960,827","296,960,827",+0.000%
1,ef_grazing_gha,"43,292,202","43,292,202",+0.000%
2,ef_forest_gha,"279,411,154","279,411,154",+0.000%
3,ef_fishing_gha,"122,282,182","122,282,182",+0.000%
4,ef_built_up_gha,"29,109,650","29,109,650",+0.000%
5,ef_carbon_gha,"1,464,806,974","1,464,806,974",+0.000%
6,ef_total_gha,"2,235,862,989","2,235,862,989",-0.000%
7,bc_cropland_gha,"350,150,978","350,150,978",+0.000%
8,bc_grazing_gha,"133,403,121","133,403,121",-0.000%
9,bc_forest_gha,"401,794,614","401,794,614",+0.000%


In [35]:
# Summary for additional countries: China, India, Brazil
demo_countries = {
    231: "USA",
    41: "China, mainland",
    100: "India",
    21: "Brazil",
}

multi = results[
    (results["area_code"].isin(demo_countries.keys()))
    & (results["year"] == DEMO_YEAR)
][
    ["area_code", "area", "year",
     "ef_cropland_gha", "ef_grazing_gha", "ef_forest_gha",
     "ef_fishing_gha", "ef_built_up_gha", "ef_carbon_gha",
     "ef_total_gha", "bc_total_gha", "ecological_deficit_gha",
     "population", "ef_per_capita_gha", "bc_per_capita_gha"]
].copy()

print(f"Comparison: Multiple countries ({DEMO_YEAR})")
display(
    multi.style.format({
        col: "{:,.0f}" for col in multi.columns
        if col not in ["area_code", "area", "year", "ef_per_capita_gha", "bc_per_capita_gha"]
    }).format({"ef_per_capita_gha": "{:.2f}", "bc_per_capita_gha": "{:.2f}"})
)

Comparison: Multiple countries (2023)


,area_code,area,year,ef_cropland_gha,ef_grazing_gha,ef_forest_gha,ef_fishing_gha,ef_built_up_gha,ef_carbon_gha,ef_total_gha,bc_total_gha,ecological_deficit_gha,population,ef_per_capita_gha,bc_per_capita_gha
199,21,Brazil,2023,174422947.579298,118346297.500612,226994364.640884,13892015.942657,4184268.000000,132590286.425903,670430180.089355,813969353.627100,143539173.537745,211140.729000,3.18,3.86
359,41,"China, mainland",2023,485756678.194491,79531151.279592,233546754.640884,495907354.546464,8420280.000000,3504171855.541719,4807334074.203151,1202576873.792997,-3604757200.410154,1422584.933000,3.38,0.85
849,100,India,2023,310848913.982173,174041046.035714,249765784.475138,81444127.005369,11109318.000000,918661270.236613,1745870459.735007,399284548.026475,-1346585911.708532,1438069.596000,1.21,0.28
2049,231,United States of America,2023,296960827.319610,43292201.632653,279411153.977901,122282182.254305,29109650.240000,1464806973.848070,2235862989.272539,928615027.825781,-1307247961.446758,343477.335000,6.51,2.70


In [36]:
# Global summary: sum all countries -> Number of Earths, Overshoot Day
gs_2023 = global_summary[global_summary["year"] == DEMO_YEAR].iloc[0]

# Recompute from country data
all_2023 = results[results["year"] == DEMO_YEAR]
world_ef = all_2023["ef_total_gha"].sum()
world_bc = all_2023["bc_total_gha"].sum()
n_earths = world_ef / world_bc
overshoot_day = math.floor(365 * world_bc / world_ef)

from datetime import datetime, timedelta
overshoot_date = datetime(DEMO_YEAR, 1, 1) + timedelta(days=overshoot_day - 1)

print("=" * 70)
print(f"GLOBAL SUMMARY ({DEMO_YEAR})")
print("=" * 70)
print(f"World EF:          {world_ef / 1e9:>10.2f} billion gha")
print(f"World BC:          {world_bc / 1e9:>10.2f} billion gha")
print(f"Overshoot:         {(world_ef - world_bc) / 1e9:>10.2f} billion gha")
print(f"Number of Earths:  {n_earths:>10.2f}")
print(f"Overshoot Day:     {overshoot_date.strftime('%B %d')} (day {overshoot_day})")
print(f"\nStored values:     {gs_2023['number_of_earths']:.2f} Earths, day {int(gs_2023['overshoot_day'])}")
print(f"Match: {int(gs_2023['overshoot_day']) == overshoot_day}")

GLOBAL SUMMARY (2023)
World EF:               24.56 billion gha
World BC:               11.84 billion gha
Overshoot:              12.71 billion gha
Number of Earths:        2.07
Overshoot Day:     June 25 (day 176)

Stored values:     2.07 Earths, day 176
Match: True
